<a href="https://colab.research.google.com/github/Karthik5412/News-Categorizer-Nlp-Model/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [99]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

In [100]:
df = pd.read_csv('/content/drive/MyDrive/news model/english_news_dataset.csv')

In [101]:
df.head(2)

,Headline,Content,News Categories,Date
0,Congress leader Baljinder Singh shot dead at h...,Congress leader Baljinder Singh was shot dead ...,['national'],19-09-2023
1,17-year-old girl preparing for NEET dies by su...,Another NEET aspirant died by suicide in Rajas...,['national'],19-09-2023


In [102]:
df.shape

(307696, 4)

In [103]:
df.isna().sum()

,0
Headline,0
Content,0
News Categories,0
Date,0


In [104]:
import re

df['News Categories'] = df['News Categories'].apply(lambda x : re.sub(f'[^a-zAz\s]','',x.lower())).str.split(' ')

In [105]:
df['News Categories'].value_counts()

,count
News Categories,
[entertainment],16728
[miscellaneous],13691
[science],12465
"[politics, national]",12165
[sports],11348
...,...
"[miscellaneous, world, sports]",1
"[experiment, sports, olympics]",1
"[olympics, national]",1


In [106]:
needed_cols = ['business', 'world', 'entertainment', 'politics', 'automobile', 'national','science', 'sports','education',
               'startup','healthfitness','technology', 'travel','fasion']

def refeared_final (x):
    if x[0] in needed_cols:
        return x[0]
    else :
        for i in x:
            if i in (needed_cols) :
                return i
        else :
            return ''


df['Final cat'] = df['News Categories'].apply(refeared_final)
df = df[df['Final cat'] != '']
df['Final cat'].value_counts()

,count
Final cat,
business,38412
world,36794
entertainment,30452
politics,28656
national,23350
sports,22125
automobile,21903
science,20875
education,20410


In [107]:
df[df['Final cat'] == 'miscellaneous']

,Headline,Content,News Categories,Date,Final cat


In [108]:
# concatinating columns
df['Description'] = df['Headline'] + ' ' + df['Content']
df.head()

,Headline,Content,News Categories,Date,Final cat,Description
0,Congress leader Baljinder Singh shot dead at h...,Congress leader Baljinder Singh was shot dead ...,[national],19-09-2023,national,Congress leader Baljinder Singh shot dead at h...
1,17-year-old girl preparing for NEET dies by su...,Another NEET aspirant died by suicide in Rajas...,[national],19-09-2023,national,17-year-old girl preparing for NEET dies by su...
2,Hampers to welcome MPs in new Parliament tomor...,In order to mark the first-ever working day of...,[national],19-09-2023,national,Hampers to welcome MPs in new Parliament tomor...
3,"Only 10% women lawmakers in RS, while only 14%...","Congress President Mallikarjun Kharge, while s...",[national],19-09-2023,national,"Only 10% women lawmakers in RS, while only 14%..."
4,"Ganesh temple decorated with notes, coins wort...",The Sri Sathya Ganapathi Temple in Bengaluru a...,[national],19-09-2023,national,"Ganesh temple decorated with notes, coins wort..."


In [109]:
# Droping unwanted columns
df = df.drop(columns= ['News Categories', 'Date', 'Headline', 'Content'])
df = df[df['Final cat'] != 'miscellaneous']

df = df[['Description', 'Final cat']]
df.head()

,Description,Final cat
0,Congress leader Baljinder Singh shot dead at h...,national
1,17-year-old girl preparing for NEET dies by su...,national
2,Hampers to welcome MPs in new Parliament tomor...,national
3,"Only 10% women lawmakers in RS, while only 14%...",national
4,"Ganesh temple decorated with notes, coins wort...",national


In [110]:
df['Final cat'].value_counts()

,count
Final cat,
business,38412
world,36794
entertainment,30452
politics,28656
national,23350
sports,22125
automobile,21903
science,20875
education,20410


In [111]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [112]:
stopwords = set(stopwords.words('english'))
lemetizer = WordNetLemmatizer()

In [113]:
df['Description'][12]

'14-year-old girl dies, several others hospitalised after eating shawarma in TN A 14-year-old schoolgirl died after eating chicken shawarma bought from a hotel in Tamil Nadu. Thirteen medical college students were also reportedly hospitalised on the same night after eating shawarma from the same hotel. Namakkal Collector S Uma ordered the destruction of all food items and issued a notice to the hotel, which was then sealed.'

In [114]:
# droping punctuations and numbers after converting it to lower case

import re


def refining(x):
   data = re.sub(r'[^a-zA-z\s]','',x.lower())

   data = data.split()

   cleaned = [w for w in data if w not in stopwords]

   cleaned = [lemetizer.lemmatize(w) for w in cleaned]


   return " ".join(cleaned)

df['Description'] = df['Description'].apply(refining)

df['Description'][12]

'yearold girl dy several others hospitalised eating shawarma tn yearold schoolgirl died eating chicken shawarma bought hotel tamil nadu thirteen medical college student also reportedly hospitalised night eating shawarma hotel namakkal collector uma ordered destruction food item issued notice hotel sealed'

In [115]:
from sklearn.feature_extraction.text import  TfidfVectorizer
tfidf = TfidfVectorizer(max_features= 1000, ngram_range= (1,2), sublinear_tf= True)

In [116]:
word_matrix = tfidf.fit_transform(df['Description'])

In [117]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

labels = le.fit_transform(df['Final cat'])

In [118]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

In [119]:
x_train, x_test, y_train, y_test = train_test_split(word_matrix, labels, test_size= 0.3, random_state = 42)

In [120]:
# clf = LinearSVC(class_weight='balanced')

# clf.fit(x_train, y_train)

In [121]:
# pred = clf.predict(x_test)
# pred

In [122]:
# y_test

In [123]:
# from sklearn.metrics import accuracy_score, classification_report

# print(accuracy_score(y_test, pred))
# print(classification_report(y_test, pred))

In [124]:
df['Final cat'].value_counts()

,count
Final cat,
business,38412
world,36794
entertainment,30452
politics,28656
national,23350
sports,22125
automobile,21903
science,20875
education,20410


In [126]:
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV

In [127]:
linear_svc = LinearSVC(class_weight= 'balanced', dual= False)

calib = CalibratedClassifierCV(linear_svc, cv= 3)

est = [
    ('NB', MultinomialNB()),
    ('SVM', calib),
    ('LogRegg', LogisticRegression(max_iter = 1000, class_weight='balanced'))
]

clf = StackingClassifier(estimators= est, final_estimator= LogisticRegression(max_iter= 1000), n_jobs = -1, cv= 3)

In [128]:
clf.fit(x_train, y_train)

StackingClassifier(cv=3,
                   estimators=[('NB', MultinomialNB()),
                               ('SVM',
                                CalibratedClassifierCV(cv=3,
                                                       estimator=LinearSVC(class_weight='balanced',
                                                                           dual=False))),
                               ('LogRegg',
                                LogisticRegression(class_weight='balanced',
                                                   max_iter=1000))],
                   final_estimator=LogisticRegression(max_iter=1000),
                   n_jobs=-1)

In [129]:
pred = clf.predict(x_test)
pred

array([12, 12,  7, ..., 12, 10,  1])

In [130]:
y_test

array([12,  0,  7, ...,  7, 10,  1])

In [131]:
from sklearn.metrics import accuracy_score, classification_report

print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

0.8026305068963511
              precision    recall  f1-score   support

           0       0.91      0.94      0.93      6521
           1       0.75      0.76      0.75     11552
           2       0.86      0.88      0.87      6098
           3       0.89      0.89      0.89      9272
           4       0.74      0.75      0.74      3342
           5       0.69      0.67      0.68      6920
           6       0.84      0.83      0.84      8559
           7       0.83      0.86      0.84      6271
           8       0.92      0.90      0.91      6716
           9       0.73      0.64      0.68      4159
          10       0.66      0.65      0.66      2671
          11       0.85      0.89      0.87      2903
          12       0.71      0.71      0.71     10931

    accuracy                           0.80     85915
   macro avg       0.80      0.80      0.80     85915
weighted avg       0.80      0.80      0.80     85915

